# 動的計画法

## 概要

**動的計画法（Dynamic Programming, DP）** は、問題を小さな部分問題に分割し、その結果を記録・再利用することで効率的に解くアルゴリズム設計手法である。

### 適用条件

| 性質 | 説明 |
|---|---|
| **最適部分構造** | 問題の最適解が部分問題の最適解から構成できる |
| **重複する部分問題** | 同じ部分問題が複数回現れる（単純な再帰では無駄な計算が発生する） |

### 貪欲法との違い

| | 貪欲法 | 動的計画法 |
|---|---|---|
| 選択方法 | 局所最適を逐次選択 | 全部分問題の最適解を記録して利用 |
| 後戻り | しない | しない（ただし全パターンを網羅） |
| 適用範囲 | 貪欲選択性が成立する問題 | 最適部分構造 + 重複部分問題がある問題 |
| 計算量 | 一般に速い | 貪欲法より遅いが再帰より大幅に速い |

### 2つのアプローチ

- **トップダウン（メモ化再帰）**: 再帰で解きながら、計算済みの結果をキャッシュする
- **ボトムアップ（表形式）**: 小さい部分問題から順に解いてテーブルに埋める

## 例1: フィボナッチ数列

$F(n) = F(n-1) + F(n-2)$ は重複する部分問題の典型例。
単純な再帰では $O(2^n)$ かかるが、DP では $O(n)$ になる。

In [ ]:
import functools

# トップダウン（メモ化再帰）
@functools.cache
def fib_top_down(n: int) -> int:
    if n <= 1:
        return n
    return fib_top_down(n - 1) + fib_top_down(n - 2)


# ボトムアップ（表形式）
def fib_bottom_up(n: int) -> int:
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


n = 10
print(f"F({n}) = {fib_top_down(n)}  (トップダウン)")
print(f"F({n}) = {fib_bottom_up(n)}  (ボトムアップ)")

## 例2: 0/1 ナップサック問題

Q. 容量 $W$ のナップサックに重さ $w_i$・価値 $v_i$ のアイテムを詰めるとき、価値を最大化するには？
**アイテムは分割できない**（0/1: 使うか使わないか）。

分数ナップサックと異なり、貪欲法では最適解が保証されないため DP を使う。

**漸化式**

$$
dp[i][j] = \max(dp[i-1][j],\; dp[i-1][j - w_i] + v_i) \quad (j \geq w_i)
$$

- $dp[i][j]$: 最初の $i$ 個のアイテムから容量 $j$ で得られる最大価値

In [ ]:
def knapsack_01(capacity: int, items: list[tuple[int, int]]) -> int:
    """
    0/1 ナップサック問題

    Parameters
    ----------
    capacity : int
    items : list of (weight, value)

    Returns
    -------
    int
        最大価値
    """
    n = len(items)
    # dp[j]: 容量 j での最大価値（1次元配列で空間節約）
    dp = [0] * (capacity + 1)

    for weight, value in items:
        for j in range(capacity, weight - 1, -1):  # 逆順で更新（同アイテムの重複使用を防ぐ）
            dp[j] = max(dp[j], dp[j - weight] + value)

    return dp[capacity]


capacity = 50
items = [
    (10, 60),
    (20, 100),
    (30, 120),
]

print(f"最大価値: {knapsack_01(capacity, items)}")
# 分数ナップサックは 240 だが 0/1 では最適解が異なることを確認
# アイテム(10,60)+(20,100)+(20/30*30, 80) = 240 → 分割できないので
# 選択肢: (10,60)+(20,100) = 160 or (20,100)+(30,120) = 220 or (10,60)+(30,120) = 180
print("  ※ 分数ナップサックの場合は 240 になる")

## 例3: 最長共通部分列（LCS）

Q. 2つの文字列 $s$、$t$ の**最長共通部分列**（Longest Common Subsequence）の長さは？

部分列は連続でなくてもよい（例: `"ACE"` は `"ABCDE"` の部分列）。

**漸化式**

$$
dp[i][j] = \begin{cases}
dp[i-1][j-1] + 1 & (s[i] = t[j]) \\
\max(dp[i-1][j],\; dp[i][j-1]) & (s[i] \neq t[j])
\end{cases}
$$

In [ ]:
def lcs_length(s: str, t: str) -> int:
    m, n = len(s), len(t)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp[m][n]


def lcs_string(s: str, t: str) -> str:
    m, n = len(s), len(t)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    # 経路復元
    result = []
    i, j = m, n
    while i > 0 and j > 0:
        if s[i - 1] == t[j - 1]:
            result.append(s[i - 1])
            i -= 1; j -= 1
        elif dp[i - 1][j] > dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(result))


s, t = "ABCBDAB", "BDCAB"
print(f"s = {s!r}")
print(f"t = {t!r}")
print(f"LCS の長さ: {lcs_length(s, t)}")
print(f"LCS の例 : {lcs_string(s, t)!r}")